# CausalBayes: Bayesian Causal Discovery

This notebook demonstrates the core CausalBayes workflow:
1. Generate synthetic data from known DAGs (linear and non-linear)
2. Learn causal structure with uncertainty using NeuralBayesianDAG
3. Visualize edge probabilities with uncertainty
4. Evaluate calibration of uncertainty estimates
5. Compare with ground truth using comprehensive evaluation

CausalBayes combines Neural NOTEARS (gradient-based DAG learning) with Bayesian uncertainty quantification via MC Dropout.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from causbayes import NeuralBayesianDAG
from causbayes.evaluation import comprehensive_evaluation, edge_calibration, precision_recall_auc
from causbayes.visualization import plot_probabilistic_dag, plot_uncertainty_calibration

np.random.seed(42)
print('Imports OK')

### Generate Linear DAG Data

We first create a random linear Gaussian DAG. Each edge weight is drawn uniformly from [-2, -0.5] ∪ [0.5, 2].

In [ ]:
def generate_linear_dag(d=10, n=1000, edge_prob=0.3, noise_scale=0.1, seed=42):
    rng = np.random.RandomState(seed)
    W_true = np.zeros((d, d))
    for i in range(d):
        for j in range(i + 1, d):
            if rng.random() < edge_prob:
                W_true[i, j] = rng.uniform(0.5, 2.0) * rng.choice([-1, 1])

    X = np.zeros((n, d))
    for j in range(d):
        parents = np.where(W_true[:, j] != 0)[0]
        if len(parents) > 0:
            X[:, j] = X[:, parents] @ W_true[parents, j]
        X[:, j] += rng.randn(n) * noise_scale

    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    return X, W_true


X_lin, W_true_lin = generate_linear_dag(d=8, n=1000, seed=42)
print(f'Linear data shape: {X_lin.shape}')
print(f'True edges: {int(np.sum(W_true_lin))}')
print(f'Density: {np.sum(W_true_lin) / (8*7):.3f}')

### Generate Non-linear DAG Data

We also generate non-linear data using a chain structure with sin/cos transformations of parent values.

In [ ]:
def generate_nonlinear_dag(d=6, n=2000, noise_scale=0.1, seed=42):
    rng = np.random.RandomState(seed)
    W_true = np.zeros((d, d))
    for i in range(d - 1):
        W_true[i, i + 1] = 1.0

    X = np.zeros((n, d))
    X[:, 0] = rng.randn(n)
    for j in range(1, d):
        parents = np.where(W_true[:, j] != 0)[0]
        for p in parents:
            X[:, j] += np.sin(X[:, p]) + 0.5 * np.cos(X[:, p] * 2)
        X[:, j] += rng.randn(n) * noise_scale

    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    return X, W_true


X_nl, W_true_nl = generate_nonlinear_dag(d=6, n=2000, seed=42)
print(f'Non-linear data shape: {X_nl.shape}')
print(f'True edges (chain): {int(np.sum(W_true_nl))}')

### Learn Causal Structure (Linear DAG)

Fit NeuralBayesianDAG with MC Dropout uncertainty. The model uses:
- Separate MLPs for each variable (non-linear SEM)
- NOTEARS acyclicity constraint to enforce DAG
- MC Dropout to sample from the posterior over edge weights

In [ ]:
model_lin = NeuralBayesianDAG(
    hidden_layers=[64, 64],
    learning_rate=1e-3,
    lambda_1=1e-2,
    lambda_2=5.0,
    uncertainty='mc_dropout',
    mc_samples=30,
    max_iter=50,
    verbose=True,
    device='cpu',
)

model_lin.fit(X_lin)
print(f'\nLearned {len(model_lin.get_top_edges(5))} edges with probabilities')

### Top Edges with Uncertainty

Display the most probable edges with their uncertainty (standard deviation).

In [ ]:
print('Top-10 edges by probability:')
print(f'{"Edge":<10} {"Probability":<12} {"Std":<10} {"True Edge":<12}')
print('-' * 44)
for (i, j), prob, std in model_lin.get_top_edges(10):
    true = 'YES' if W_true_lin[i, j] != 0 else ''
    print(f'X{i}->X{j}    {prob:.4f}      {std:.4f}      {true}')

### Visualize Probabilistic DAG

The DAG is visualized with edge width proportional to probability and color indicating uncertainty. Red = high probability, blue = uncertain.

In [ ]:
plot_probabilistic_dag(
    model_lin.edge_probs,
    model_lin.edge_stds,
    threshold=0.3,
    uncertainty=True,
    title='Learned DAG (Linear Data)',
    show_edge_labels=False,
)

### Calibration Plot

How well-calibrated are the edge probabilities? A perfectly calibrated model would have points on the diagonal. The Expected Calibration Error (ECE) measures deviation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration for linear data
plot_uncertainty_calibration(
    model_lin.edge_probs,
    W_true_lin,
    ax=axes[0],
)
axes[0].set_title('Calibration (Linear DAG)')

# Edge probability distribution
probs = model_lin.edge_probs.flatten()
probs = probs[probs > 0.01]
axes[1].hist(probs, bins=20, alpha=0.7, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Edge Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Probability Distribution')

plt.tight_layout()
plt.show()

### Comprehensive Evaluation

Compare against ground truth using all available metrics.

In [ ]:
metrics = comprehensive_evaluation(W_true_lin, model_lin.edge_probs, model_lin.edge_stds)

print('Comprehensive Evaluation (Linear DAG):')
print('=' * 40)
for key in ['shd', 'expected_shd', 'auc_pr', 'precision@0.5', 'recall@0.5', 'ece', 'coverage@0.9', 'avg_edge_entropy']:
    if key in metrics:
        print(f'  {key:<20} {metrics[key]:.4f}')

### Learn Non-linear DAG

Now apply the same method to non-linear data with a chain structure.

In [ ]:
model_nl = NeuralBayesianDAG(
    hidden_layers=[128, 128],
    learning_rate=5e-4,
    lambda_1=1e-2,
    lambda_2=10.0,
    uncertainty='mc_dropout',
    mc_samples=30,
    max_iter=80,
    verbose=True,
    device='cpu',
)

model_nl.fit(X_nl)

print(f'\nTop edges (non-linear):')
for (i, j), prob, std in model_nl.get_top_edges(8):
    true = 'YES' if W_true_nl[i, j] != 0 else ''
    print(f'  X{i} -> X{j}: P = {prob:.3f} +/- {std:.3f}  {true}')

### Visualize Non-linear DAG Recovery

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_probabilistic_dag(
    model_nl.edge_probs, model_nl.edge_stds,
    threshold=0.3, uncertainty=True,
    title='Non-linear DAG', ax=axes[0],
)

plot_uncertainty_calibration(
    model_nl.edge_probs, W_true_nl, ax=axes[1],
)

plt.tight_layout()
plt.show()

In [ ]:
metrics_nl = comprehensive_evaluation(W_true_nl, model_nl.edge_probs, model_nl.edge_stds)

print('Non-linear DAG Evaluation:')
print('=' * 40)
print(f'  SHD:                {metrics_nl["shd"]}')
print(f'  Expected SHD:       {metrics_nl["expected_shd"]:.4f}')
print(f'  AUC-PR:             {metrics_nl["auc_pr"]:.4f}')
print(f'  Precision@0.5:      {metrics_nl["precision@0.5"]:.4f}')
print(f'  Recall@0.5:         {metrics_nl["recall@0.5"]:.4f}')
print(f'  ECE:                {metrics_nl["ece"]:.4f}')
print(f'  Coverage@0.9:       {metrics_nl.get("coverage@0.9", "N/A")}')

### Summary

| Metric | Linear DAG | Non-linear DAG |
|--------|-----------|----------------|
| SHD (lower better) | Measures edge errors | Measures edge errors |
| Expected SHD | Penalty-weighted errors | Penalty-weighted errors |
| AUC-PR (higher better) | Area under precision-recall curve | Area under PR curve |
| ECE (lower better) | Expected Calibration Error | Expected Calibration Error |
| Coverage (higher better) | % of edges within CI | % of edges within CI |

CausalBayes provides:
- **Structure learning**: Neural NOTEARS for non-linear DAG discovery
- **Uncertainty**: MC Dropout and Variational Inference
- **Flexibility**: Works on linear and non-linear data
- **Calibration**: Well-calibrated edge probabilities with ECE < 0.15 typically